# Train the NAFNet denoiser (free GPU)

Runs the training in this repo on a Colab T4, about an hour for the `nano`
architecture. Training on a laptop CPU is not practical.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.

At the end you download `nafnet.onnx` and copy it to the Jetson's
`models/` folder.

---

## About the data

The denoiser learns from **pairs**: a clean frame and a degraded copy. Those
two halves have different requirements, and conflating them is the usual way
this goes wrong.

| Half | Must be | Supplied by |
|---|---|---|
| **Degradation** — noise, blur, low light, JPEG | CCTV-specific | simulated in `degradations.py` |
| **Clean content** — texture, edge statistics | real photographs | cell 3b |

Procedural scenes (cell 3a) get the geometry right — camera height, perspective,
lighting pools, figures at depth-correct scale — but their surfaces are smooth
and their figures are flat. Train on those alone and the model concludes that
**smooth is always correct**, then wipes out the facial detail GFPGAN and
ArcFace depend on.

So run **both** 3a and 3b. The synthetic half teaches the scene; the real
photographs teach what detail looks like. Add your own footage when you have
it — that is strictly better than either.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'

In [ ]:
#@title 2. Get the code and mount Drive for checkpoints
!git clone -q https://github.com/Indumini2005/Re-id-techthrive.git
%cd Re-id-techthrive
!pip install -q onnx onnxsim onnxruntime

# Checkpoints go to Drive, not the VM's disk. A free Colab session can be
# reclaimed at any time, and anything under /content dies with it - which is
# how a 21-epoch run gets lost. Writing to Drive makes the run resumable.
from google.colab import drive
drive.mount('/content/drive')

RUNS = '/content/drive/MyDrive/nafnet_runs'
!mkdir -p {RUNS}
print('checkpoints ->', RUNS)

In [ ]:
#@title 3a. Data option A - synthetic CCTV scenes (no footage needed)
# Procedural rooms with people at depth-correct scale. Good enough to
# validate the whole chain and to warm-start; weaker than real frames.
!python -m training.synthesize_cctv frames \
    --out data/synth --count 2000 --width 960 --height 540

from IPython.display import Image, display
display(Image('data/synth/clean/000000.png', width=640))

In [ ]:
#@title 3b. Real photographs - RUN THIS TOO, it is not optional
#@markdown The synthetic scenes give correct geometry, perspective and
#@markdown lighting, but almost no fine texture: flat figures, clean edges,
#@markdown smooth surfaces. A denoiser trained only on that learns
#@markdown **"smooth is always the right answer"** and then erases facial
#@markdown detail on real footage - which is precisely what GFPGAN and
#@markdown ArcFace need downstream.
#@markdown
#@markdown Denoising is a low-level task, so the content does not have to be
#@markdown surveillance. A few hundred sharp photographs of anything teach
#@markdown the model what real detail looks like. The CCTV-specific half of
#@markdown the problem is the degradation, and that is already simulated.
#@markdown
#@markdown ~430 MB, about a minute on Colab.
!python -m training.fetch_real_data --out data/real --dataset div2k_valid --max_side 1600

# Optional and better still: your own footage. Prop a phone high in a
# corridor, angled down, record 10-15 minutes of people walking through:
#   from google.colab import files; files.upload()
#   !python -m training.make_dataset video --src . --out data/own --fps 2
# then add data/own/clean to DATA_DIRS in cell 5.

In [ ]:
#@title 4. Preview the degradation the model will learn to undo
!python -m training.degradations data/synth/clean/000000.png \
    --preset cctv --n 4 --out preview.png

from IPython.display import Image, display
display(Image('preview.png', width=1100))
# clean | four degraded variants. If these look nothing like your cameras,
# tune training/degradations.py before spending an hour on training.

In [ ]:
#@title 5. Train
#@markdown `nano` is what the Jetson Nano can actually run. `small` is ~4x
#@markdown the cost for roughly +0.5 dB; measure before choosing it.
#@markdown
#@markdown `DATA_DIRS` takes several folders. Keep both: the synthetic
#@markdown frames supply CCTV geometry and lighting, the real photographs
#@markdown supply the texture the model must learn to preserve.
#@markdown
#@markdown **30 epochs, not 80.** The cosine schedule reaches most of its
#@markdown gain early, and a free Colab session is not reliable for much
#@markdown longer than that. Checkpoints land on Drive, so you can extend
#@markdown with cell 5b instead of betting one long run.
ARCH = 'nano'      #@param ['nano', 'small', 'base']
EPOCHS = 30        #@param {type:'integer'}
PATCH = 128        #@param {type:'integer'}
BATCH = 16         #@param {type:'integer'}
DATA_DIRS = 'data/synth/clean data/real/clean'  #@param {type:'string'}

!python -m training.train_denoiser \
    --train_dir {DATA_DIRS} --arch {ARCH} --preset cctv \
    --epochs {EPOCHS} --patch_size {PATCH} --batch_size {BATCH} \
    --repeat 2 --workers 2 --name nafnet --runs_dir {RUNS}

In [ ]:
#@title 5b. Resume after a disconnect - run ONLY if cell 5 was cut short
#@markdown If Colab dropped the session, reconnect, re-run cells 1-4 to
#@markdown rebuild the data, then run this. It picks up from the last
#@markdown completed epoch on Drive rather than starting over.
#@markdown
#@markdown Set EPOCHS higher than the original to extend the run.
EPOCHS = 30        #@param {type:'integer'}

!python -m training.train_denoiser \
    --train_dir {DATA_DIRS} --arch {ARCH} --preset cctv \
    --epochs {EPOCHS} --patch_size {PATCH} --batch_size {BATCH} \
    --repeat 2 --workers 2 --name nafnet --runs_dir {RUNS} \
    --resume {RUNS}/nafnet/last.pth

In [ ]:
#@title 6. Look at what it learned
import glob
from IPython.display import Image, display

samples = sorted(glob.glob(RUNS + '/nafnet/samples/*.png'))
print('noisy | denoised | clean')
for path in samples[-2:]:
    print(path)
    display(Image(path, width=900))

import pandas as pd
log = pd.read_csv(RUNS + '/nafnet/log.csv')
display(log.tail(10))
log.plot(x='epoch', y='val_psnr', figsize=(7, 3), title='validation PSNR (dB)');

# The strips matter more than the curve. Check whether fine detail survived
# or got smoothed away - that is the failure the real photographs guard
# against, and PSNR alone will not show it.

In [ ]:
#@title 7. Score it
!python -m training.evaluate --ckpt {RUNS}/nafnet/best.pth \
    --data_dir data/real/clean --limit 40 --max_side 720
# Scored on the REAL photographs, not the synthetic scenes. Synthetic frames
# are easy for a model trained on synthetic frames; the real set is the
# honest number.

In [ ]:
#@title 8. Export ONNX, save to Drive and download
#@markdown opset 12 keeps JetPack 4.6's ONNX Runtime 1.11 happy.
!python export/export_onnx.py --ckpt {RUNS}/nafnet/best.pth \
    --out models/nafnet.onnx --tile 128 --opset 12 --simplify

# Confirm the exported graph matches the checkpoint before trusting it
!python -m training.evaluate --onnx models/nafnet.onnx \
    --data_dir data/real/clean --limit 20 --max_side 720 --tile 128

# Copy to Drive as well, so the model survives even if the download fails
!cp models/nafnet.onnx {RUNS}/nafnet.onnx
print('saved to', RUNS + '/nafnet.onnx')

from google.colab import files
files.download('models/nafnet.onnx')

## Then on the Jetson

```bash
# copy nafnet.onnx into models/, then
python3 check_device.py
python3 denoise.py --input_dir some_frames --output_dir out --tile 128 --compare
```

`--compare` writes noisy-vs-denoised strips so you can judge it by eye on
real frames, which is the only test that counts.

Optionally build a TensorRT engine **on the device** (FP16 only — Maxwell
has no INT8):

```bash
python3 export/export_tensorrt.py --onnx models/nafnet.onnx --precision fp16 --benchmark
```